# Lab 7 – Noun-Phrase Chunking with a Hidden Markov Model

**Tasks**
1. Use the **Forward algorithm** to compute the probability of every test sentence.
2. Use **Viterbi decoding** to predict the NP tags (`B-NP`, `I-NP`, `O`) and evaluate with `seqeval`.

Zero counts are handled with **Add-K smoothing**.

---
## 1. Theory

### 1.1 The chunking problem
Noun-phrase chunking is a sequence-labelling task. Each word gets one of three tags:

| Tag | Meaning |
|---|---|
| `B-NP` | first word of a noun phrase |
| `I-NP` | word inside (continuing) a noun phrase |
| `O` | word not in any noun phrase |

Example: `Rockwell/B-NP International/I-NP Corp./I-NP 's/B-NP Tulsa/I-NP unit/I-NP said/O`

### 1.2 Hidden Markov Model (HMM)
An HMM is a generative model with **hidden states** (here the tags) and **observations** (here the words).
It is defined by $\lambda = (\pi, A, B)$:

* **States** $S=\{B, I, O\}$, $N=3$
* **Initial distribution** $\pi_i = P(q_1 = s_i)$
* **Transition matrix** $A_{ij} = P(q_t = s_j \mid q_{t-1} = s_i)$
* **Emission matrix** $B_j(w) = P(o_t = w \mid q_t = s_j)$

Two assumptions make it tractable:
1. **Markov assumption:** $P(q_t \mid q_1..q_{t-1}) = P(q_t \mid q_{t-1})$
2. **Output independence:** $P(o_t \mid q_1..q_T, o_1..o_{t-1}) = P(o_t \mid q_t)$

The joint probability of a word sequence $O=o_1..o_T$ and tag sequence $Q=q_1..q_T$ is

$$P(O,Q)=\pi_{q_1}B_{q_1}(o_1)\prod_{t=2}^{T}A_{q_{t-1}q_t}\,B_{q_t}(o_t)$$

### 1.3 Parameter estimation (supervised MLE)
Because the training data is tagged, we just count:

$$\pi_i=\frac{C(\text{sentence starts with } s_i)}{\#\text{sentences}},\quad
A_{ij}=\frac{C(s_i\to s_j)}{C(s_i\to\cdot)},\quad
B_j(w)=\frac{C(s_j,w)}{C(s_j)}$$

### 1.4 Add-K smoothing
Unseen events would get probability 0 and kill the whole product. Add-K adds a pseudo-count $K$ to every event:

$$A_{ij}=\frac{C(s_i\to s_j)+K}{C(s_i\to\cdot)+K\,N},\qquad
B_j(w)=\frac{C(s_j,w)+K}{C(s_j)+K\,(|V|+1)},\qquad
\pi_i=\frac{C(\text{start}=s_i)+K}{\#\text{sent}+K\,N}$$

$|V|+1$ accounts for the vocabulary plus one special `<UNK>` token. Every word not seen in training is mapped to `<UNK>`, so it
gets probability $K/(C(s_j)+K(|V|+1))$ under each tag (nonzero, and larger for frequent tags).
We pick $K$ on a held-out development split.

### 1.5 Forward algorithm – $P(O\mid\lambda)$ (Task 1)
The probability of a sentence is the sum of the joint probability over **all** $N^T$ tag sequences:
$P(O)=\sum_Q P(O,Q)$. Brute force is exponential; dynamic programming makes it $O(N^2T)$.

Define $\alpha_t(j)=P(o_1..o_t,\;q_t=s_j\mid\lambda)$.

* Initialisation: $\alpha_1(j)=\pi_j\,B_j(o_1)$
* Recursion: $\alpha_t(j)=\Big[\sum_{i=1}^{N}\alpha_{t-1}(i)\,A_{ij}\Big]B_j(o_t)$
* Termination: $P(O\mid\lambda)=\sum_{j=1}^{N}\alpha_T(j)$

**Numerical issue:** multiplying many small numbers underflows (a 40-word sentence is about $10^{-100}$ or smaller). We work in **log space**
and replace the sum by `logsumexp`:
$\log\alpha_t(j)=\log B_j(o_t)+\operatorname{logsumexp}_i\big(\log\alpha_{t-1}(i)+\log A_{ij}\big)$.

### 1.6 Viterbi algorithm – best tag sequence (Task 2)
Decoding asks for $Q^*=\arg\max_Q P(Q\mid O)=\arg\max_Q P(O,Q)$. Viterbi is the forward algorithm with **max** instead of **sum**,
plus back-pointers.

Define $\delta_t(j)=\max_{q_1..q_{t-1}}P(o_1..o_t,\;q_1..q_{t-1},\;q_t=s_j)$.

* Initialisation: $\delta_1(j)=\pi_jB_j(o_1)$, $\psi_1(j)=0$
* Recursion: $\delta_t(j)=\max_i[\delta_{t-1}(i)A_{ij}]\,B_j(o_t)$, $\psi_t(j)=\arg\max_i[\delta_{t-1}(i)A_{ij}]$
* Termination: $q_T^*=\arg\max_j\delta_T(j)$
* Backtrace: $q_t^*=\psi_{t+1}(q_{t+1}^*)$

Again computed in log space (sums of logs, no underflow).
Note that $\max_Q P(O,Q)\le P(O)$ always, which gives us a handy sanity check.

### 1.7 Evaluation with `seqeval`
`seqeval` converts tag sequences into **entity spans** (a `B-NP` followed by `I-NP`s) and scores at the *chunk* level, so a chunk only counts
as correct if both boundaries are exactly right:

$$P=\frac{TP}{TP+FP},\quad R=\frac{TP}{TP+FN},\quad F_1=\frac{2PR}{P+R}$$

---
## 2. Implementation

In [3]:
# !pip install seqeval scipy numpy   # uncomment if running on a fresh environment
import numpy as np
from collections import Counter
from scipy.special import logsumexp
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

TRAIN_PATH = "Data/train_NP.txt"
TEST_PATH  = "Data/test_NP.txt"

### 2.1 Load the data
Format: one `word<TAB>tag` per line, blank line between sentences.

In [4]:
def read_conll(path):
    sents, tags = [], []
    w, t = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if w:
                    sents.append(w); tags.append(t); w, t = [], []
                continue
            parts = line.split("\t") if "\t" in line else line.split()
            w.append(parts[0]); t.append(parts[-1])
    if w:
        sents.append(w); tags.append(t)
    return sents, tags

train_x, train_y = read_conll(TRAIN_PATH)
test_x,  test_y  = read_conll(TEST_PATH)

print("train sentences:", len(train_x), "| tokens:", sum(map(len, train_x)))
print("test  sentences:", len(test_x),  "| tokens:", sum(map(len, test_x)))
print("tag distribution (train):", Counter(t for s in train_y for t in s))
print("\nexample:", list(zip(train_x[0], train_y[0])))

train sentences: 823 | tokens: 19172
test  sentences: 77 | tokens: 1896
tag distribution (train): Counter({'O': 8394, 'I-NP': 5727, 'B-NP': 5051})

example: [('Rockwell', 'B-NP'), ('International', 'I-NP'), ('Corp.', 'I-NP'), ("'s", 'B-NP'), ('Tulsa', 'I-NP'), ('unit', 'I-NP'), ('said', 'O'), ('it', 'B-NP'), ('signed', 'O'), ('a', 'B-NP'), ('tentative', 'I-NP'), ('agreement', 'I-NP'), ('extending', 'O'), ('its', 'B-NP'), ('contract', 'I-NP'), ('with', 'O'), ('Boeing', 'B-NP'), ('Co.', 'I-NP'), ('to', 'O'), ('provide', 'O'), ('structural', 'B-NP'), ('parts', 'I-NP'), ('for', 'O'), ('Boeing', 'B-NP'), ("'s", 'B-NP'), ('747', 'I-NP'), ('jetliners', 'I-NP'), ('.', 'O')]


### 2.2 HMM with Add-K smoothing
We lowercase words (reduces sparsity) and map unseen words to `<UNK>`.

In [5]:
TAGS = ["B-NP", "I-NP", "O"]
tag2id = {t: i for i, t in enumerate(TAGS)}
UNK = "<UNK>"

class HMM:
    def __init__(self, k=0.01, lowercase=True):
        self.k = k
        self.lowercase = lowercase

    def _norm(self, w):
        return w.lower() if self.lowercase else w

    def fit(self, X, Y):
        N = len(TAGS)
        self.vocab = {UNK: 0}
        for sent in X:
            for w in sent:
                self.vocab.setdefault(self._norm(w), len(self.vocab))
        V = len(self.vocab)

        c_start = np.zeros(N)
        c_trans = np.zeros((N, N))
        c_emit  = np.zeros((N, V))
        for words, tags in zip(X, Y):
            ids = [tag2id[t] for t in tags]
            c_start[ids[0]] += 1
            for a, b in zip(ids[:-1], ids[1:]):
                c_trans[a, b] += 1
            for w, s in zip(words, ids):
                c_emit[s, self.vocab[self._norm(w)]] += 1

        k = self.k
        self.log_pi = np.log((c_start + k) / (c_start.sum() + k * N))
        self.log_A  = np.log((c_trans + k) / (c_trans.sum(1, keepdims=True) + k * N))
        # <UNK> column gets only the pseudo-count k (it never occurs in training)
        self.log_B  = np.log((c_emit + k) / (c_emit.sum(1, keepdims=True) + k * V))
        return self

    def encode(self, words):
        return [self.vocab.get(self._norm(w), 0) for w in words]

    # ---------- Task 1: Forward algorithm (log space) ----------
    def forward(self, words):
        o = self.encode(words)
        T, N = len(o), len(TAGS)
        log_alpha = np.empty((T, N))
        log_alpha[0] = self.log_pi + self.log_B[:, o[0]]
        for t in range(1, T):
            # sum over previous states i:  logsumexp_i( alpha_{t-1}(i) + log A[i,j] )
            log_alpha[t] = logsumexp(log_alpha[t-1][:, None] + self.log_A, axis=0) + self.log_B[:, o[t]]
        return logsumexp(log_alpha[-1]), log_alpha      # log P(O), full table

    # ---------- Task 2: Viterbi decoding (log space) ----------
    def viterbi(self, words):
        o = self.encode(words)
        T, N = len(o), len(TAGS)
        delta = np.empty((T, N))
        psi   = np.zeros((T, N), dtype=int)
        delta[0] = self.log_pi + self.log_B[:, o[0]]
        for t in range(1, T):
            scores = delta[t-1][:, None] + self.log_A       # scores[i, j]
            psi[t]   = scores.argmax(axis=0)
            delta[t] = scores.max(axis=0) + self.log_B[:, o[t]]
        path = [int(delta[-1].argmax())]
        for t in range(T - 1, 0, -1):
            path.append(int(psi[t, path[-1]]))
        path.reverse()
        return [TAGS[i] for i in path], delta[-1].max()    # tags, log P(O, Q*)

### 2.3 Choose K on a development split
We hold out the last 10 % of training sentences, try several values of K, and keep the one with the best chunk-level F1.

In [6]:
split = int(0.9 * len(train_x))
tr_x, tr_y = train_x[:split], train_y[:split]
dv_x, dv_y = train_x[split:], train_y[split:]

results = []
for k in [1e-5, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0]:
    m = HMM(k=k).fit(tr_x, tr_y)
    pred = [m.viterbi(s)[0] for s in dv_x]
    results.append((k, f1_score(dv_y, pred)))
    print(f"K = {k:<8g} dev F1 = {results[-1][1]:.4f}")

best_k = max(results, key=lambda r: r[1])[0]
print("\nBest K:", best_k)

K = 1e-05    dev F1 = 0.7006
K = 0.0001   dev F1 = 0.7006
K = 0.001    dev F1 = 0.7020
K = 0.01     dev F1 = 0.7093
K = 0.05     dev F1 = 0.7275
K = 0.1      dev F1 = 0.7393
K = 0.5      dev F1 = 0.7387
K = 1        dev F1 = 0.6998

Best K: 0.1


### 2.4 Train the final model on the full training set

In [7]:
model = HMM(k=best_k).fit(train_x, train_y)
print("vocab size:", len(model.vocab))

print("\nInitial probabilities  P(tag at start):")
for t, p in zip(TAGS, np.exp(model.log_pi)): print(f"  {t:5s} {p:.4f}")

print("\nTransition matrix  P(next | prev):")
print("        " + "  ".join(f"{t:>6s}" for t in TAGS))
for t, row in zip(TAGS, np.exp(model.log_A)):
    print(f"{t:6s}  " + "  ".join(f"{p:6.4f}" for p in row))

vocab size: 4256

Initial probabilities  P(tag at start):
  B-NP  0.6572
  I-NP  0.0001
  O     0.3426

Transition matrix  P(next | prev):
          B-NP    I-NP       O
B-NP    0.0295  0.6956  0.2749
I-NP    0.0459  0.3875  0.5666
O       0.5403  0.0000  0.4597


Notice `O -> I-NP` and the `I-NP`/`B-NP` structure in the transition matrix. An `I-NP` should almost never directly follow `O`,
and the HMM learns this from data, which is what lets Viterbi produce well-formed chunks.

### 2.5 Sanity check: Forward vs brute force
On a short sentence we enumerate all $3^T$ tag sequences and verify that the forward result equals the sum, and that Viterbi's best path is the argmax.

In [8]:
import itertools

def joint_logprob(m, words, tag_ids):
    o = m.encode(words)
    lp = m.log_pi[tag_ids[0]] + m.log_B[tag_ids[0], o[0]]
    for t in range(1, len(o)):
        lp += m.log_A[tag_ids[t-1], tag_ids[t]] + m.log_B[tag_ids[t], o[t]]
    return lp

short = next(s for s in test_x if 4 <= len(s) <= 7)
all_lp = {q: joint_logprob(model, short, q) for q in itertools.product(range(3), repeat=len(short))}

brute_forward = logsumexp(list(all_lp.values()))
brute_best_q  = max(all_lp, key=all_lp.get)

fwd_lp, _ = model.forward(short)
vit_tags, vit_lp = model.viterbi(short)

print("sentence      :", " ".join(short))
print("forward  logP :", fwd_lp)
print("brute    logP :", brute_forward)
print("viterbi tags  :", vit_tags)
print("brute best    :", [TAGS[i] for i in brute_best_q])
assert np.isclose(fwd_lp, brute_forward) and vit_tags == [TAGS[i] for i in brute_best_q]
print("\nOK: forward == brute-force sum and Viterbi == brute-force argmax")

sentence      : Friday 's Market Activity
forward  logP : -29.618644028003036
brute    logP : -29.61864402800304
viterbi tags  : ['B-NP', 'B-NP', 'I-NP', 'I-NP']
brute best    : ['B-NP', 'B-NP', 'I-NP', 'I-NP']

OK: forward == brute-force sum and Viterbi == brute-force argmax


## 3. Task 1 – Probability of every test sentence (Forward algorithm)

In [9]:
import pandas as pd

rows = []
for i, s in enumerate(test_x):
    logp, _ = model.forward(s)
    rows.append({
        "id": i,
        "length": len(s),
        "log P(O)": logp,
        "log10 P(O)": logp / np.log(10),
        "P(O)": float(np.exp(logp)),          # may underflow to 0.0 for long sentences, hence the log columns
        "sentence": " ".join(s),
    })
fwd_df = pd.DataFrame(rows)
fwd_df.to_csv("test_sentence_probabilities.csv", index=False)

pd.set_option("display.max_colwidth", 70)
display(fwd_df.head(10))
print("Sentences whose raw probability underflows to 0.0 in float64:", int((fwd_df['P(O)'] == 0).sum()), "/", len(fwd_df))

,id,length,log P(O),log10 P(O),P(O),sentence
0,0,37,-270.451254,-117.455487,3.503585e-118,Confidence in the pound is widely expected to take another sharp d...
1,1,27,-197.257980,-85.668052,2.147571e-86,Chancellor of the Exchequer Nigel Lawson 's restated commitment to...
2,2,29,-219.315166,-95.247367,5.657615e-96,But analysts reckon underlying support for sterling has been erode...
3,3,36,-239.277237,-103.916784,1.211202e-104,This has increased the risk of the government being forced to incr...
4,4,31,-222.809333,-96.764864,1.718448e-97,`` The risks for sterling of a bad trade figure are very heavily o...
5,5,36,-254.538703,-110.544754,2.852632e-111,"`` If there is another bad trade number , there could be an awful ..."
6,6,39,-272.537419,-118.361497,4.350136e-119,"Forecasts for the trade figures range widely , but few economists ..."
7,7,25,-183.396073,-79.647903,2.249559e-80,The August deficit and the # 2.2 billion gap registered in July ar...
8,8,28,-208.775528,-90.670060,2.137667e-91,"Sanjay Joshi , European economist at Baring Brothers & Co. , said ..."
9,9,24,-171.127872,-74.319890,4.787508e-75,"At the same time , he remains fairly pessimistic about the outlook..."


Sentences whose raw probability underflows to 0.0 in float64: 0 / 77


Longer sentences have lower probability; this is expected as $P(O)$ is a product over positions. The average log-probability per word is therefore a fairer way to compare sentences of different length:

In [10]:
fwd_df["logP per word"] = fwd_df["log P(O)"] / fwd_df["length"]
print(fwd_df[["length", "log P(O)", "logP per word"]].describe().round(3))
print("\nCorrelation(length, log P):", round(fwd_df["length"].corr(fwd_df["log P(O)"]), 3))

       length  log P(O)  logP per word
count  77.000    77.000         77.000
mean   24.623  -173.250         -7.061
std     9.488    66.751          0.475
min     4.000  -343.351         -8.285
25%    18.000  -215.099         -7.336
50%    25.000  -170.895         -7.115
75%    30.000  -126.416         -6.802
max    50.000   -29.619         -5.630

Correlation(length, log P): -0.988


## 4. Task 2 – Viterbi decoding and evaluation with `seqeval`

In [11]:
pred_y, vit_scores = [], []
for s in test_x:
    tags, lp = model.viterbi(s)
    pred_y.append(tags); vit_scores.append(lp)

# Viterbi path probability can never exceed the total sentence probability (forward)
assert all(v <= f + 1e-9 for v, f in zip(vit_scores, fwd_df["log P(O)"]))

print(classification_report(test_y, pred_y, digits=4))
print(f"Chunk-level  precision: {precision_score(test_y, pred_y):.4f}")
print(f"Chunk-level  recall   : {recall_score(test_y, pred_y):.4f}")
print(f"Chunk-level  F1       : {f1_score(test_y, pred_y):.4f}")
print(f"Token-level  accuracy : {accuracy_score(test_y, pred_y):.4f}")

              precision    recall  f1-score   support

          NP     0.6754    0.7579    0.7143       475

   micro avg     0.6754    0.7579    0.7143       475
   macro avg     0.6754    0.7579    0.7143       475
weighted avg     0.6754    0.7579    0.7143       475

Chunk-level  precision: 0.6754
Chunk-level  recall   : 0.7579
Chunk-level  F1       : 0.7143
Token-level  accuracy : 0.8650


### 4.1 Token-level confusion matrix and examples

In [12]:
gold_flat = [t for s in test_y for t in s]
pred_flat = [t for s in pred_y for t in s]
conf = pd.crosstab(pd.Series(gold_flat, name="gold"), pd.Series(pred_flat, name="pred"))
display(conf.reindex(index=TAGS, columns=TAGS, fill_value=0))

print("\nSample predictions (word / gold / pred, '*' marks errors):")
for i in range(3):
    print(f"\nSentence {i}   log P = {fwd_df.loc[i, 'log P(O)']:.2f}")
    for w, g, p in zip(test_x[i], test_y[i], pred_y[i]):
        print(f"  {w:18s} {g:5s} {p:5s} {'' if g == p else '*'}")

pred,B-NP,I-NP,O
gold,,,
B-NP,412,32,31
I-NP,47,490,42
O,74,30,738



Sample predictions (word / gold / pred, '*' marks errors):

Sentence 0   log P = -270.45
  Confidence         B-NP  B-NP  
  in                 O     O     
  the                B-NP  B-NP  
  pound              I-NP  I-NP  
  is                 O     O     
  widely             O     B-NP  *
  expected           O     O     
  to                 O     O     
  take               O     O     
  another            B-NP  B-NP  
  sharp              I-NP  I-NP  
  dive               I-NP  I-NP  
  if                 O     O     
  trade              B-NP  B-NP  
  figures            I-NP  I-NP  
  for                O     O     
  September          B-NP  B-NP  
  ,                  O     O     
  due                O     O     
  for                O     O     
  release            B-NP  B-NP  
  tomorrow           B-NP  I-NP  *
  ,                  O     O     
  fail               O     B-NP  *
  to                 O     O     
  show               O     O     
  a                  B-

### 4.2 Optional: effect of case-folding and unknown words

In [13]:
unk_tokens = sum(w.lower() not in model.vocab for s in test_x for w in s)
total_tokens = sum(len(s) for s in test_x)
print(f"Unknown test tokens: {unk_tokens}/{total_tokens} ({100*unk_tokens/total_tokens:.2f}%)")

m_case = HMM(k=best_k, lowercase=False).fit(train_x, train_y)
pred_case = [m_case.viterbi(s)[0] for s in test_x]
print(f"F1 with lowercase=True : {f1_score(test_y, pred_y):.4f}")
print(f"F1 with lowercase=False: {f1_score(test_y, pred_case):.4f}")

Unknown test tokens: 345/1896 (18.20%)
F1 with lowercase=True : 0.7143
F1 with lowercase=False: 0.7122
